In [0]:
dbutils.fs.ls("/Volumes/workspace/lab_tuan1/raw/")



In [0]:
CATALOG, SCHEMA = "workspace", "lab_tuan1"
TBL = f"{CATALOG}.{SCHEMA}.bronze_orders_tuan1"

In [0]:
path = "/Volumes/workspace/lab_tuan1/raw/orders_batch1.csv"

df_raw = spark.read.csv(path, header=True, inferSchema=False) 
df_raw.printSchema()
df_raw.show(5, truncate=False)


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.lab_tuan1.bronze_orders_tuan1 (
  order_id    STRING,
  shop_code   STRING,
  order_ts    TIMESTAMP,     
  amount      DECIMAL(18,2),  -- KHÔNG dùng DOUBLE cho tiền
  quantity    INT,
  product_id  STRING,
  status      STRING,
  channel     STRING,
  updated_at  TIMESTAMP
);


In [0]:
%sql
INSERT INTO workspace.lab_tuan1.bronze_orders_tuan1
SELECT
  order_id,
  shop_code,
  coalesce(
    try_to_timestamp(order_ts, 'yyyy-MM-dd HH:mm:ss'),
    try_to_timestamp(order_ts, 'dd/MM/yyyy HH:mm')
  ) AS order_ts,
  cast(amount AS DECIMAL(18,2)),
  cast(quantity AS INT),
  product_id,
  status,
  channel,
  to_timestamp(updated_at, 'yyyy-MM-dd HH:mm:ss')
FROM read_files(
  '/Volumes/workspace/lab_tuan1/raw/orders_batch1.csv',
  format => 'csv', header => true
);


In [0]:
%sql
SELECT COUNT(*) AS so_dong, COUNT(DISTINCT order_id) AS so_order
FROM workspace.lab_tuan1.bronze_orders_tuan1;
